In [1]:
# !pip install sentence-transformers

In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TranslationPipeline
from sentence_transformers import SentenceTransformer, util
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import numpy as np

zeroshot_classifier = pipeline('zero-shot-classification', model='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7')
# model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
model = SentenceTransformer("dragonkue/BGE-m3-ko")
ko_generator = pipeline('text-generation', model='Bllossom/llama-3.2-Korean-Bllossom-3B')



Device set to use cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\parks\anaconda3\envs\nlp_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\parks\.cache\huggingface\hub\models--dragonkue--BGE-m3-ko. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


In [ ]:
def zero_shot_classification(text):
    
    candidate_labels = [
    # "업무 요청",
    # "업무 감사",
    "일상 안부",
    "사적 관심"
    ]

    zero_shot = zeroshot_classifier(text, candidate_labels, hypothesis_template="이 문장은 {}에 해당한다.")

    if zero_shot['labels'][0] == '사적 관심':
        print("관심이 있는 것 같습니다.")
        print("\n 자세한 분석을 위해 다음 단계로 넘어갑니다.")

        return zero_shot['sequence']

    else:
        print("관심이 감지되지 않았습니다.")
        return None


In [ ]:
def second_sentence_check(text):

    target_data = [
    "오빠 오늘 시간 어때요?",
    "주말에 시간 나면 같이 밥 먹을래요?",
    "이번 주말에 영화 보러 갈래요? 제가 예매할게요!",
    "언제 시간 내서 단둘이 술 한잔해요.",
    
    "나랑 사귈래?",
    "오늘따라 왜 이렇게 예쁘게 하고 왔어요?",
    "핸드크림 바를래?",
    "너랑 같이 있으니까 시간이 너무 빨리 가는 거 같아.",
    
    "자요? 갑자기 목소리 듣고 싶어서",
    "어제 누구랑 술 마셨어? 질투 나려 그러네 ㅋㅋㅋ",
    "이상형이 어떻게 되세요? 저 같은 스타일은 어때요?",
    "오늘 하루 종일 네 생각 났어."
    ]
    
    nontarget_data = [
    "네 확인해주셔서 감사합니다.",
    "업무 협조에 감사드립니다.",
    "수고하셨습니다. 조심히 들어가세요.",
    
    "오늘 하루도 즐겁게 보내세요.",
    "날씨가 많이 추워졌네요. 감기 조심하세요.",
    "식사는 맛있게 하셨나요?",
    "주말 편안하게 잘 쉬시길 바랍니다.",
    
    "도움 주셔서 정말 감사합니다.",
    "네, 알겠습니다. 신경 써주셔서 고맙습니다.",
    "좋은 정보 감사합니다. 잘 참고하겠습니다."
    ]
    
    text_vec = model.encode([text])
    target_vecs = model.encode(target_data)
    nontarget_vecs = model.encode(nontarget_data)

    target_score = np.max(cosine_similarity(text_vec, target_vecs))
    nontarget_score = np.max(cosine_similarity(text_vec, nontarget_vecs))
    
    print(f" 💌 플러팅 일치율: {target_score:.4f} vs 👨‍🦱 일상 대화 일치율: {nontarget_score:.4f}")
    
    if target_score > nontarget_score - 0.3 : 
        print("\n 플러팅이 감지되었으므로, 철벽 문구를 생성합니다.")
        return text
    else:
        print("✅ 사적 대화이지만, 이정도는 괜찮습니다. ")
        return None

In [34]:
text = input()

final_text = zero_shot_classification(text)

if final_text is not None:
    final_result = second_sentence_check(final_text)
    
    if final_result is not None:
        print("\n 철벽 문구를 생성하고 있습니다... \n")
        
        prompt =  f"상황: 애인이 있는 상태에서 다른 사람의 호감을 거절함.\n상대방: {final_result}\n 단호한 단답형 거절:"
        
        generated_output = ko_generator(
            prompt,
            max_new_tokens=30,
            pad_token_id=ko_generator.tokenizer.eos_token_id,       
            truncation=True,    # 길면 자른다.
            num_return_sequences=1, # 답변은 하나만
            repetition_penalty=1.5, # 헛소리 금지
            temperature=0.1 # 최대한 정적인 답변
        )
        
        full_text = generated_output[0]['generated_text']
        full_text = full_text.split("단호한 단답형 거절:")[1].split('\n')[0].strip()
        print(full_text)

관심이 있는 것 같습니다.

 자세한 분석을 위해 다음 단계로 넘어갑니다.
 💌 플러팅 일치율: 0.4465 vs 👨‍🦱 일상 대화 일치율: 0.3793
플러팅이 감지되었으므로, 철벽 문구를 생성합니다.

 철벽 문구를 생성하고 있습니다... 

"아니요, 안 되겠습니다."


In [33]:
generated_output

[{'generated_text': '상황: 애인이 있는 상태에서 다른 사람의 호감을 거절함.\n상대방: 나랑 사귀자\n 단답형 거절: "아, 그게 아니야. 우리가 서로 잘 어울리는 사람이 아니라서." 또는 "나도 같은 생각이야."\n 대화 형'}]